In [74]:
from tqdm import tqdm
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# 构造数据

## 环境变量向量

$$
\begin{align*}

e &:\ [w, r, n, v, m^{\text{<ack>}}] \\
e &:\ \text{environmental variable vector} \\
w &:\ \text{weather condition}  \in \{ 1, 2, 3, 4, 5, 6, 7, 8, 9, 10 \} \\
r &:\ \text{road condition}  \in \{ 1, 2, 3, 4, 5, 6, 7, 8, 9, 10 \} \\
n &:\ \text{surrounding vehicles condition} \in \{ 1, 2, 3, 4, 5, 6, 7, 8, 9, 10 \} \\
v &:\ \text{vehicle speed} \in \{ 1, 2, 3, 4, 5, 6, 7, 8, 9, 10 \}

\end{align*}
$$


$$
\text{hazard level } h = \begin{cases}
    1 & \text{lowest accident risk} \\
    10 & \text{highest accident risk}
\end{cases}
$$


$$
\text{ACK message } m^{\text{<ack>}} = \begin{cases}
    1 & \text{ACK received (no retransmission)} \\
    2 & \text{ACK not received (retransmission required)}
\end{cases}
$$

### 正态分布
$$
\begin{align*}

\mathcal{X} \sim \mathcal{N}(\mu, \sigma^{2})

\end{align*}
$$

#### 密度函数
$$
\begin{align*}

\mathcal{f(x)} = \frac{1}{\sigma \sqrt{2 \pi}} \exp{
    \left(
        -\frac{(x - \mu)^{2}}{2 \sigma^{2}}
    \right)
}

\end{align*}
$$

$$
\begin{align*}

w = \mathrm{Clamp}_{[1, 10]} \left(
        \mathrm{Round} \left(
            \mathcal{N}(5.5, 2^2)
        \right)
    \right)

\end{align*}
$$

In [75]:
def generate_weather_condition(batch_size, device = "cpu"):
    w = torch.normal(5.5, 2.0, size = (batch_size, ), device = device)
    return torch.clamp(w.round(), 1, 10)

$$
\begin{align*}

r_{\mathrm{prior}} &= \mathcal{N}(5.5, 1^{2}) \\

r &= \mathrm{Clamp}_{[1, 10]} \left(
    \mathrm{Round} \left(
        r_{\mathrm{prior}} + 0.5(w - 5.5) + \mathcal{N}(0, 1^{2})
    \right)
\right)

\end{align*}
$$

In [76]:
def generate_road_condition(w):
    prior = torch.normal(5.5, 1.0, size = (w.shape[0], ), device = w.device)
    noise = torch.normal(0, 1.0, size = (w.shape[0], ), device = w.device)
    r = prior + 0.5 * (w - 5.5) + noise
    return torch.clamp(r.round(), 1, 10)

$$
\begin{align*}

d_{\mathrm{prior}} &= r_{min} + r_{\max} - r = 11 - r \\

d &= \mathrm{Clamp}_{[1, 10]} \left( d_{\mathrm{prior}} + \mathcal{N}(0, 1^{2}) \right)

\end{align*}
$$


In [77]:
def generate_vehicle_density(r):
    prior = 11.0 - r
    noise = torch.normal(0, 1.0, size = (r.shape[0], ), device = r.device)
    d = prior + noise
    return torch.clamp(d, 1, 10)

### 高斯函数
$$
\begin{align*}

\mathcal{f(x)} = A \exp{
    \left(
        -\frac{(x - \mu)^{2}}{2\sigma^{2}}
    \right)
}

\end{align*}
$$

$$
\begin{align*}

n_{\mathrm{prior}} &= A \exp{
    \left(
        -\frac{(d - \mu)^{2}}{2 \sigma^{2}}
    \right)
} \\

 &= 10 \cdot \exp \left( 
    -\frac{(d - 5.5)^{2}}{2 \cdot 2^{2}}
\right)

\end{align*}
$$

$$
n = \mathrm{Clamp}_{[1, 10]} \left(
    \mathrm{Round} \left(
        n_{\mathrm{prior}} + \mathcal{N}(0, 1^{2})
    \right)
\right)
$$

In [78]:
def generate_surrounding_vehicles_condition(d):
    prior = 10.0 * torch.exp(-((d - 5.5) ** 2) / (2 * (2.0 ** 2)))
    noise = torch.normal(0, 1.0, size = (d.shape[0], ), device = d.device)
    n = prior + noise
    return torch.clamp(n.round(), 1, 10)

$$
\begin{align*}

V &\sim 
\begin{cases}
\mathcal{N}(0, 1) & &u < 0.70 \\
\mathcal{N}(0, 3) & 0.70 \le &u < 0.95 \\
\mathcal{N}(0, 6) & &u \ge 0.95
\end{cases}

\quad u \sim U(0,1)

\quad v_{\text{prior}} = |V| \\

v_{\mathrm{d}} &= A \exp{
    \left(
        -\frac{(d - \mu)^{2}}{2 \sigma^{2}}
    \right)
} = 2 \cdot \exp \left( 
    -\frac{(d - 5.5)^{2}}{2 \cdot 2^{2}}
\right) \\

v &= \mathrm{Clamp}_{[1, 10]} \left(
    \mathrm{Round} \left(
        v_{\mathrm{prior}} + v_{\mathrm{d}} + \mathcal{N}(0, 1^{2})
    \right)
\right)

\end{align*} 
$$

In [79]:
def generate_v_prior(batch_size, device = "cpu"):
    u = torch.rand(batch_size, device = device)
    mask1 = u < 0.70
    mask2 = (u >= 0.70) & (u < 0.95)
    mask3 = u >= 0.95

    v = torch.empty(batch_size, device = device)
    v[mask1] = torch.normal(0, 1.0, size = (mask1.sum(), ), device = device)
    v[mask2] = torch.normal(0, 3.0, size = (mask2.sum(), ), device = device)
    v[mask3] = torch.normal(0, 6.0, size = (mask3.sum(), ), device = device)

    return v.abs()

In [80]:
def generate_vehicle_speed(d):
    prior = generate_v_prior(d.shape[0], device = d.device)
    v_d_p = 2.0 * torch.exp(-((d - 5.5) ** 2) / (2 * (2.0 ** 2)))
    noise = torch.normal(0, 1.0, size = (d.shape[0], ), device = d.device)
    
    v = prior + v_d_p + noise
    return torch.clamp(v.round(), 1, 10)

### received signal to interference plus noise ratio(SINR)
$$
\begin{align*}

\eta = \frac{P_{R}}{\sigma_{I} + \sigma_{N}}

\end{align*}
$$

$$
\begin{align*}

P_{R} &= \alpha(w_{\max} + w_{\min} - w) + \beta(d_{\max} + d_{\min} - d) + \mathcal{N}(0, 1^{2}) \\

&= \alpha(11 - w) + \beta(11 - d) + \mathcal{N}(0, 1^{2}) \\

&= 1.2(11 - w) + 0.8(11 - d) + \mathcal{N}(0, 1^{2}) \\
\\
\sigma_{I} &= \beta_{1}d + \mathcal{N}(0, 1) = 0.4d + \mathcal{N}(0, 1) \\
\sigma_{N} &= \sigma_{0} + \mathcal{N}(0, 0.1) = 1 + \mathcal{N}(0, 0.1) \\

\end{align*}
$$

In [81]:
def generate_SINR(w, d):
    P_R = (1.2 * (11 - w) + 0.8 * (11 - d) + torch.normal(0, 1.0, size = (w.shape[0], ), device = w.device)).relu()
    sigma_I = (0.4 * d + torch.normal(0, 1.0, size = (w.shape[0], ), device = w.device)).relu()
    sigma_N = (1.0 + torch.normal(0, 0.1, size = (w.shape[0], ), device = w.device)).relu()
    SINR = P_R / (sigma_I + sigma_N + 1e-6)
    return SINR

In [82]:
def generate_channel_condition(SINR):
    SINR_db = 10 * torch.log10(SINR + 1e-6)
    c = torch.floor((SINR_db + 5) / 2.5) + 1
    return torch.clamp(c, 1, 10).long()

$$
\begin{align*}

l_{action} = l_{\text{optimal}} = 
\begin{cases}
1 & c \in \{ 1, 2 \} \\
2 & c \in \{ 3, 4, 5 \} \\
3 & c \in \{ 6, 7, 8 \} \\
4 & c \in \{ 9, 10 \}
\end{cases} 

\end{align*}
$$

In [83]:
def generate_communication_rate(c):
    l = torch.ones_like(c)
    l = torch.where((3 <= c) & (c <= 5), torch.tensor(2, device = c.device), l)
    l = torch.where((6 <= c) & (c <= 8), torch.tensor(3, device = c.device), l)
    l = torch.where(c >= 9, torch.tensor(4, device = c.device), l)
    return l

$$
\begin{align*}

m^{\text{<ack>}} = 
\begin{cases}
1 & l_{\text{action}} \leq l_{\text{optimal}} \\
2 & l_{\text{action}} > l_{\text{optimal}}
\end{cases}

\end{align*}
$$

In [84]:
def generate_m_ack(l_action, l_optimal):
    success = (l_action <= l_optimal).long()
    ones = torch.ones_like(success)
    twos = torch.full_like(success, 2)
    return torch.where(success == 1, ones, twos)

In [85]:
def generate_env_c(batch_size, device = "cpu"):
    w = generate_weather_condition(batch_size, device)
    r = generate_road_condition(w)
    d = generate_vehicle_density(r)
    n = generate_surrounding_vehicles_condition(d)
    v = generate_vehicle_speed(d)

    SINR1 = generate_SINR(w, d)
    SINR2 = generate_SINR(w, d)

    c1 = generate_channel_condition(SINR1)
    c2 = generate_channel_condition(SINR2)

    l_action = generate_communication_rate(c1)
    l_optimal = generate_communication_rate(c2)

    m_ack = generate_m_ack(l_action, l_optimal)

    return torch.stack([w, r, n, v, m_ack], dim = 1), c1

## 状态

$$
\begin{align*}

s &:\ [e, c, q, m] \\
s &:\ \text{state} \\
e &:\ \text{environmental variable vector} \\
c &:\ \text{channel condition}  \in \{ 1, 2, 3, 4, 5, 6, 7, 8, 9, 10 \} \\
q &:\ \text{status of the DENM message queue, indicates the age of the oldest DENM message in the current queue.} \in \{ 1, 2, 3, 4 \} \\
m &:\ \text{message} \in \{ 0, 1, 2, 3 \} \\

\end{align*}
$$

$$
\begin{align*}

p_{u} &= \tau_{w}^{(w)}p_{w}^{(w)} + \tau_{r}^{(r)}p_{r}^{(r)} + \tau_{n}^{(n)}p_{n}^{(n)} + \tau_{v}^{(v)}p_{v}^{(v)}

\end{align*}
$$

In [86]:
def compute_p_u(env):
    w = env[:, 0]
    r = env[:, 1]
    n = env[:, 2]
    v = env[:, 3]

    w_norm = (w - 1) / 9.0
    r_norm = (r - 1) / 9.0
    n_norm = (n - 1) / 9.0
    v_norm = (v - 1) / 9.0

    tau_w = 0.25
    tau_r = 0.25
    tau_n = 0.30
    tau_v = 0.20

    p_u = tau_w*w_norm + tau_r*r_norm + tau_n*n_norm + tau_v*v_norm
    return p_u

$$
\begin{aligned}

p_{\text{shifted}} &= \begin{cases}
0              &       p_{u} + \delta  <  0 \\
p_{u} + \delta & 0 \le p_{u} + \delta \le 1 \\
1              & 1  <  p_{u} + \delta
\end{cases} \\

q' &= 1 + \lfloor 
    (q_{\max} - 1)(p_{\text{shifted}})^{\alpha} 
\rfloor \\

q &= \begin{cases}
1        &       q'  <  1        \\
q'       & 1 \le q' \le q_{\max} \\
q_{\max} &       q'  >  q_{\max}
\end{cases}

\end{aligned}
$$

In [87]:
def generate_q(p_u, q_max = 4, alpha = 3.0, delta = 0.05):
    p_shifted = torch.clamp(p_u + delta, 0, 1)
    q = 1 + torch.floor((q_max - 1) * torch.pow(p_shifted, alpha))
    return q.clamp(1, q_max).long()

$$
\begin{align*}

m = 
\begin{cases}
0 & \text{No message} \\
1 & \text{CAM message} \\
2 & \text{DENM message} \\
3 & \text{CAM and DENM message entering the message queue at this time}
\end{cases}

\end{align*}
$$

In [ ]:
def generate_m(p_u, cam_frec = 0.7):
    p_m0 = (1.0 - cam_frec) * (1.0 - p_u)
    p_m1 = cam_frec * (1.0 - p_u)
    p_m2 = (1.0 - cam_frec) * p_u
    p_m3 = cam_frec * p_u
    
    probs = torch.stack([p_m0, p_m1, p_m2, p_m3], dim = -1)
    m = torch.multinomial(probs, num_samples = 1)
    return m.squeeze(1)

In [ ]:
def generate_state_accident_prob(batch_size, device="cpu"):
    e, c = generate_env_c(batch_size, device) # e: [B,5], c: [B]
    p_u = compute_p_u(e)                      # [B]
    q = generate_q(p_u)                       # [B]
    m = generate_m(p_u)                       # [B]

    c = c.unsqueeze(1)  # [B, 1]
    q = q.unsqueeze(1)  # [B, 1]
    m = m.unsqueeze(1)  # [B, 1]

    state = torch.cat([e, c, q, m], dim = 1) # [B,8]

    return state, p_u

# ADMM-Net

$$
\begin{align*}

\textbf{X}&\textbf{-update (Low-rank Module)} \\

\mathcal{D}_{\tau}(X) &= \mathrm{Conv2D} \left( \mathrm{ReLU_{\tau} \left( \mathrm{Conv2D}(X) \right) } \right) \\

\mathrm{X}^{(k + 1)} &= \mathrm{Conv2D} \left( \mathrm{ReLU} \left( \mathrm{Conv2D} \left( Z^{(k)} - U^{(k)} \right) - \frac{\rho}{2} \right) \right)

\end{align*}
$$

In [90]:
class UpdateBlockX(nn.Module):
    def __init__(self, conv1, conv2, init_tau = 0.1):
        super().__init__()
        self.conv1 = conv1
        self.conv2 = conv2
        self.tau = nn.Parameter(torch.tensor(init_tau, dtype = torch.float32))  # ρ/2

    # [B, 1, 1, F] -> [B, h, 1, F] -> [B, 1, 1, F]
    def forward(self, Z0_minus_U0):
        return self.conv2(F.relu(self.conv1(Z0_minus_U0) - self.tau))

$$
\begin{align*}

\textbf{Z}&\textbf{-update (Threshold Module)} \\
T &= X^{(k+1)} - U^{(k)} + Z^{(k)} \\
Z^{(k+1)} &= Soft_{\tau}(T) = sign(T) · \mathrm{ReLU} \left( |T| - \frac{\lambda}{\rho} \right) \\

\end{align*}
$$

In [91]:
class UpdateBlockZ(nn.Module):
    def __init__(self, init_tau = 0.1):
        super().__init__()
        self.tau = nn.Parameter(torch.tensor(init_tau, dtype = torch.float32))  # λ/ρ

    def forward(self, X1, Z0_minus_U0):
        T = X1 + Z0_minus_U0
        
        return torch.sign(T) * F.relu(torch.abs(T) - self.tau)

$$
\begin{align*}

\textbf{U-update}&\textbf{ (Multiplier Module)} \\
U^{(k+1)} &= U^{(k)} + X^{(k+1)} - Z^{(k+1)} \\

\end{align*}
$$


In [92]:
class UpdateBlockU(nn.Module):
    def forward(self, U0, X1, Z1):
        return U0 + X1 - Z1

In [93]:
class AdmmBlock(nn.Module):
    def __init__(self, conv1, conv2, init_tau_x = 0.1, init_tau_z = 0.1):
        super().__init__()
        self.x_updater = UpdateBlockX(conv1, conv2, init_tau = init_tau_x)
        self.z_updater = UpdateBlockZ(init_tau = init_tau_z)
        self.u_updater = UpdateBlockU()

    def forward(self, Z0, U0):
        Z0_minus_U0 = Z0 - U0
        X1 = self.x_updater(Z0_minus_U0)
        Z1 = self.z_updater(X1, Z0_minus_U0)
        U1 = self.u_updater(U0, X1, Z1)
        
        return X1, Z1, U1

In [94]:
class AdmmNet(nn.Module):
    def __init__(self,iter_count = 6, conv_channel_cnt = 8, init_tau_x = 0.1, init_tau_z = 0.1):
        super().__init__()
        self.iter_count = iter_count

        self.shared_conv1 = nn.Conv2d(
            in_channels = 1,
            out_channels = conv_channel_cnt,
            kernel_size = 3,
            padding = 1,
            bias = True
        )

        self.shared_conv2 = nn.Conv2d(
            in_channels = conv_channel_cnt,
            out_channels = 1,
            kernel_size = 3,
            padding = 1,
            bias = True
        )

        blocks = []
        for _ in range(iter_count):
            blocks.append(
                AdmmBlock(
                    conv1 = self.shared_conv1,
                    conv2 = self.shared_conv2,
                    init_tau_x = init_tau_x,
                    init_tau_z = init_tau_z
                )
            )

        self.blocks = nn.ModuleList(blocks)

    def forward(self, Z0, U0):
        Z, U = Z0, U0
        X = None
        for block in self.blocks:
            X, Z, U = block(Z, U)

        return X, Z, U

In [95]:
def save_admm_net(model, path):
    torch.save(model.state_dict(), path)
    print(f"已保存 ADMM-Net 模型至: {path}")

def load_admm_net(config, path):
    iter_count = config["iter_count"]
    init_tau_x = config["init_tau_x"]
    init_tau_z = config["init_tau_z"]
    conv_channel_cnt = config["conv_channel_cnt"]
    
    model = AdmmNet(iter_count, conv_channel_cnt, init_tau_x, init_tau_z)
    state_dict = torch.load(path, map_location = torch.device("cpu"))
    model.load_state_dict(state_dict)
    
    print(f"已加载 ADMM-Net 模型自: {path}")
    
    return model

## 损失函数

$$
\begin{align*}

L_{\delta}(x) = \begin{cases}
    
    \frac{1}{2} x^{2}               & |x| \le \delta \\

    \delta(|x| - \frac{1}{2}\delta) & |x|  >  \delta \\

\end{cases}

\quad \delta = 1.0

\end{align*}
$$

In [ ]:
def calc_huber_loss(tgt, pred, mask, delta = 1.0):
    x = pred[mask] - tgt[mask]
    x_abs_val = x.abs()
    quadratic = 0.5 * x.pow(2)
    linear = delta * (x_abs_val - 0.5 * delta)
    loss = torch.where(x_abs_val <= delta, quadratic, linear)
    return loss.mean(), x.numel()

$$
\begin{align*}
S &\in \mathbb{R}^{m \times n} \\
n &: 特征数量                   \\
m &: 样本数量                   \\
\end{align*}
$$

特征空间
$$
\begin{align*}

    A    &= S^{T}S \in \mathbb{R}^{n \times n}        \\

a_{i, j} &= \sum_{k = 1}^{m}(S_{k, i} \cdot S_{k, j}) \\

\end{align*}
$$

特征向量
$$
\begin{align*}

Av_{i} = \lambda_{i}v_{i}



\end{align*}
$$



$$
\begin{align*}

AV &= \Lambda V \\

\left[ \begin{array}{cccc}
a_{11} & a_{12} & \cdots & a_{1n} \\
a_{21} & a_{22} & \cdots & a_{2n} \\
\vdots & \vdots &        & \vdots \\
a_{n1} & a_{n2} & \cdots & a_{nn} \\
\end{array} \right]

\left[ \begin{array}{cccc}
v_{11} & v_{12} & \cdots & v_{1n} \\
v_{21} & v_{22} & \cdots & v_{2n} \\
\vdots & \vdots &        & \vdots \\
v_{n1} & v_{n2} & \cdots & v_{nn} \\
\end{array} \right]

&=

\left[ \begin{array}{cccc}
\lambda_{11} &        0     & \cdots &        0     \\
       0     & \lambda_{22} & \cdots &        0     \\
    \vdots   &     \vdots   &        &     \vdots   \\
       0     &        0     & \cdots & \lambda_{nn} \\
\end{array} \right]

\left[ \begin{array}{cccc}
v_{11} & v_{12} & \cdots & v_{1n} \\
v_{21} & v_{22} & \cdots & v_{2n} \\
\vdots & \vdots &        & \vdots \\
v_{n1} & v_{n2} & \cdots & v_{nn} \\
\end{array} \right]               \\

\left[ \begin{array}{cccc}
\sum_{i = 1}^{n}(a_{1i}v_{i1})   &   \sum_{i = 1}^{n}(a_{1i}v_{i2})   &   \cdots   &   \sum_{i = 1}^{n}(a_{1i}v_{in}) \\
\sum_{i = 1}^{n}(a_{2i}v_{i1})   &   \sum_{i = 1}^{n}(a_{2i}v_{i2})   &   \cdots   &   \sum_{i = 1}^{n}(a_{2i}v_{in}) \\
            \vdots               &               \vdots               &            &               \vdots             \\
\sum_{i = 1}^{n}(a_{ni}v_{i1})   &   \sum_{i = 1}^{n}(a_{ni}v_{i2})   &   \cdots   &   \sum_{i = 1}^{n}(a_{ni}v_{in}) \\
\end{array} \right]

&=

\left[ \begin{array}{cccc}
\lambda_{11}v_{11} & \lambda_{11}v_{12} & \cdots & \lambda_{11}v_{1n} \\
\lambda_{22}v_{21} & \lambda_{22}v_{22} & \cdots & \lambda_{22}v_{2n} \\
        \vdots     &         \vdots     &        &         \vdots     \\
\lambda_{nn}v_{n1} & \lambda_{nn}v_{n2} & \cdots & \lambda_{nn}v_{nn} \\
\end{array} \right]

\end{align*}
$$

$$
\begin{align*}

AV &= \Lambda V \\

\left[ \begin{array}{cccc}
a_{11} & a_{12} & \cdots & a_{1n} \\
a_{21} & a_{22} & \cdots & a_{2n} \\
\vdots & \vdots &        & \vdots \\
a_{n1} & a_{n2} & \cdots & a_{nn} \\
\end{array} \right]

\left[ \begin{array}{cccc}
v_{11} & v_{12} & \cdots & v_{1n} \\
v_{21} & v_{22} & \cdots & v_{2n} \\
\vdots & \vdots &        & \vdots \\
v_{n1} & v_{n2} & \cdots & v_{nn} \\
\end{array} \right]

&=

\left[ \begin{array}{cccc}
\lambda_{11} &        0     & \cdots &        0     \\
       0     & \lambda_{22} & \cdots &        0     \\
    \vdots   &     \vdots   &        &     \vdots   \\
       0     &        0     & \cdots & \lambda_{nn} \\
\end{array} \right]

\left[ \begin{array}{cccc}
v_{11} & v_{12} & \cdots & v_{1n} \\
v_{21} & v_{22} & \cdots & v_{2n} \\
\vdots & \vdots &        & \vdots \\
v_{n1} & v_{n2} & \cdots & v_{nn} \\
\end{array} \right]               \\

\left[ \begin{array}{cccc}
\sum_{i = 1}^{n}(a_{1i}v_{i1})   &   \sum_{i = 1}^{n}(a_{1i}v_{i2})   &   \cdots   &   \sum_{i = 1}^{n}(a_{1i}v_{in}) \\
\sum_{i = 1}^{n}(a_{2i}v_{i1})   &   \sum_{i = 1}^{n}(a_{2i}v_{i2})   &   \cdots   &   \sum_{i = 1}^{n}(a_{2i}v_{in}) \\
            \vdots               &               \vdots               &            &               \vdots             \\
\sum_{i = 1}^{n}(a_{ni}v_{i1})   &   \sum_{i = 1}^{n}(a_{ni}v_{i2})   &   \cdots   &   \sum_{i = 1}^{n}(a_{ni}v_{in}) \\
\end{array} \right]

&=

\left[ \begin{array}{cccc}
\lambda_{11}v_{11} & \lambda_{11}v_{12} & \cdots & \lambda_{11}v_{1n} \\
\lambda_{22}v_{21} & \lambda_{22}v_{22} & \cdots & \lambda_{22}v_{2n} \\
        \vdots     &         \vdots     &        &         \vdots     \\
\lambda_{nn}v_{n1} & \lambda_{nn}v_{n2} & \cdots & \lambda_{nn}v_{nn} \\
\end{array} \right]

\end{align*}
$$

特征空间
$$
\begin{align*}

\left(
    \mathrm{det}(A^{T}A - \lambda I) = 0 
\right) &\Rightarrow \lambda \\

\sigma  &= \sqrt{\lambda} \\



\end{align*}
$$

$$
\begin{align*}


A &= \left[ \begin{array}{cccc}

a_{11} & a_{12} & \cdots & a_{18} \\
a_{21} & a_{22} & \cdots & a_{28} \\
\vdots & \vdots &        & \vdots \\
a_{81} & a_{82} & \cdots & a_{88} \\

\end{array} \right]               \\


U &= \left[ \begin{array}{cccc}

u_{11} & u_{12} & \cdots & u_{18} \\
u_{21} & u_{22} & \cdots & u_{28} \\
\vdots & \vdots &        & \vdots \\
u_{81} & u_{82} & \cdots & u_{88} \\

\end{array} \right]               \\


\sum &= \left[ \begin{array}{cccc}

\sigma_{1} &      0     & \cdots &      0     \\
     0     & \sigma_{2} & \cdots &      0     \\
  \vdots   &   \vdots   &        &   \vdots   \\
     0     &      0     & \cdots & \sigma_{8} \\

\end{array} \right] \quad

\sigma_{1} \ge \sigma_{2} \ge ... \ge \sigma_{8} \ge 0 \\


V^{T} &= \left[ \begin{array}{cccc}

v_{11} & v_{12} & \cdots & v_{18} \\
v_{21} & v_{22} & \cdots & v_{28} \\
\vdots & \vdots &        & \vdots \\
v_{81} & v_{82} & \cdots & v_{88} \\

\end{array} \right]


\end{align*}
$$

$$
\Large

\begin{align*}

A = U\sum V^{T} &= \left[ \begin{array}{cccc}

\sum_{i = 1}^{8}(u_{1i}\sigma_{i}v_{i1})   &   \sum_{i = 1}^{8}(u_{1i}\sigma_{i}v_{i2})   &   \cdots   &   \sum_{i = 1}^{8}(u_{1i}\sigma_{i}v_{i2}) \\
\sum_{i = 1}^{8}(u_{2i}\sigma_{i}v_{i1})   &   \sum_{i = 1}^{8}(u_{2i}\sigma_{i}v_{i2})   &   \cdots   &   \sum_{i = 1}^{8}(u_{2i}\sigma_{i}v_{i2}) \\
                  \vdots                   &                     \vdots                   &            &                     \vdots                 \\
\sum_{i = 1}^{8}(u_{8i}\sigma_{i}v_{i1})   &   \sum_{i = 1}^{8}(u_{8i}\sigma_{i}v_{i2})   &   \cdots   &   \sum_{i = 1}^{8}(u_{8i}\sigma_{i}v_{i2}) \\

\end{array} \right]                         \\


\end{align*}
$$

$$
\begin{align*}

  Z       &\in \mathbb{R}^{m \times n}  \\

  Z       &= U\sum{}V^{T}               \\

\|Z\|_{*} &= \sum_{i = 1}^{r}\sigma_{i} \\

\end{align*}
$$

In [ ]:
def calc_nuclear_norm(z):
    Z = z.view(z.size(0), 1, -1)        # 每个样本一个 1×F 矩阵
    s = torch.linalg.svdvals(Z)         # [B, 1]
    return s.sum(dim = -1).mean()         # batch 平均核范数

$$
\begin{align*}

\min_{X, Z} &\left(
    
    \sum_{i,j \in \Theta}L_{\delta}(X_{i, j} - Y_{i, j}) + \lambda\| Z \|_{*}

\right)                      \\

&\text{s.t.} \quad X - Z = U \\

\end{align*}
$$

In [ ]:
def calc_loss(tgt, pred, mask, z = None, delta = 1.0, lambda_coef = 0.0):
    huber_loss, cnt = calc_huber_loss(tgt, pred, mask, delta)

    loss = huber_loss
    if z is not None and lambda_coef > 0:
        loss = loss + lambda_coef * calc_nuclear_norm(z)

    return loss, cnt

# 训练ADMM-Net

In [98]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AdmmNet(
    iter_count = 6,
    conv_channel_cnt = 8,
    init_tau_x = 0.1,
    init_tau_z = 0.1
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr = 1e-4)

# DDQN

雷达模式(radar mode)
$$
a_{k}^{r} \\
k \in \{ 1, 2 \} \\
\text{short-range detection mode}: a_{1}^{r} \\
\text{long-range detection mode}: a_{2}^{r} \\

$$

通信模式(communication mode)
$$
a_{l}^{d}\\
d \in \{ \text{CAM}, \text{DENM} \} \\
l \in \{ 1, 2, 3, 4 \} \\
$$

In [103]:
class QNetwork(nn.Module):
    def __init__(self, state_dim, action_dim = 2 + 8, hidden_dims = [128, 128]):
        super().__init__()
        layers = []
        in_features = state_dim
        for h_features in hidden_dims:
            layers.append(nn.Linear(in_features, h_features))
            layers.append(nn.ReLU())
            in_features = h_features

        layers.append(nn.Linear(in_features, action_dim))
        self.network = nn.Sequential(*layers)

    def forward(self, state):
        return self.network(state)

In [104]:
def save_q_net(model, path):
    torch.save(model.state_dict(), path)
    print(f"已保存 Q-Network 模型至: {path}")

def load_q_net(config, path):
    state_dim = config["state_dim"]
    action_dim = config["action_dim"]
    hidden_dims = config["hidden_dims"]

    model = QNetwork(state_dim, action_dim, hidden_dims)
    state_dict = torch.load(path, map_location = torch.device("cpu"))
    model.load_state_dict(state_dict)

    print(f"已加载 Q-Network 模型自: {path}")

    return model

$$
\begin{align*}

\text{Q-VALUE} &= Q(s, a; \theta) \\

a &= \operatorname*{argmax}_{a_j \in \mathcal{A}} Q(s,a_j;\theta) \\

\end{align*}
$$


In [105]:
@torch.no_grad()
def q_vals_for_states(network, states):
    return network(states)

@torch.no_grad()
def q_vals_for_actions(q_vals, actions):
    return q_vals.gather(1, actions)

@torch.no_grad()
def actions_for_q_vals(q_vals):
    return q_vals.argmax(dim = 1, keepdim = True)

## State Transition Function

### Based on the canonical definition of Markov decision processes:
$$
\begin{align*}
\mathcal{T} &= P(s' | s, a) \\
\end{align*}
$$

### Paper:
$$
\begin{align*}
\mathcal{T} &= P(s' | p_{u}, s, n, a) \\
\end{align*}
$$

In [106]:
class StateTransitionModel(nn.Module):
    def __init__(self, state_dim, action_dim = 2 + 8, hidden_dims = [32]):
        super().__init__()
        layers = []
        in_features = state_dim + action_dim
        for h_features in hidden_dims:
            layers.append(nn.Linear(in_features, h_features))
            layers.append(nn.ReLU())
            in_features = h_features

        layers.append(nn.Linear(in_features, state_dim))
        self.network = nn.Sequential(*layers)

    def forward(self, state, action_onehot):
        return self.network(torch.cat([state, action_onehot], dim = -1))

## Reward

$$
r_{com}(t) = 
\begin{cases}

\alpha_{1}l & a = a_{l}^{d}, m = 1, X_{t} = 0 \\

-\alpha_{2}l & a = a_{l}^{d}, X_{t} = 1 \\

0 & a = a_{k}^{r} \\

\end{cases}
$$


In [107]:
def com_reward(alpha1, alpha2, l, is_comm_action, is_cam_message, is_accident):
    is_comm_action = is_comm_action.float()
    is_cam_message = is_cam_message.float()
    is_accident    = is_accident.float()

    reward_cam_normal = alpha1 * l * is_comm_action * is_cam_message * (1 - is_accident)
    reward_comm_accident = -alpha2 * l * is_comm_action * is_accident

    return reward_cam_normal + reward_comm_accident

$$
r_{rad}(t) = 
\begin{cases}

-\beta_{1}(1 - p_{u}(t)) & a = a_{k}^{r}, X_{t} = 0 \\

\beta_{2}(1 - p_{k}(t)) & a = a_{k}^{r}, X_{t} = 1 \\

\end{cases}
$$


In [108]:
def rad_reward(beta1, p_u, beta2, p_k, is_radar_action, is_accident):
    is_radar_action = is_radar_action.float()
    is_accident = is_accident.float()

    reward_no_accident = -beta1 * (1.0 - p_u) * is_radar_action * (1 - is_accident)
    reward_accident = beta2 * (1.0 - p_k) * is_radar_action * is_accident

    return reward_no_accident + reward_accident

In [109]:
# delta_i

$$
r_{age}(t) =
\begin{cases}

q_{\max} - \delta_{i} & a = a_{l}^{\text{DENM}}, \\

0 & a \neq a_{l}^{\text{DENM}} \\

\end{cases}
$$


In [110]:
def age_reward(q_max, delta_i, is_denm_action):
    is_denm_action = is_denm_action.float()
    return (q_max - delta_i) * is_denm_action

$$
\begin{align*}
R(t) &= \alpha r_{com}(t) + \beta r_{rad}(t) + \lambda r_{age}(t) \\
\end{align*}
$$

In [111]:
def reward(alpha, r_com, beta, r_rad, param_lambda, r_age):
    return alpha * r_com + beta * r_rad + param_lambda * r_age 

In [112]:
# class Environment:
#     def __init__(self):
        

$$
\begin{align*}
y &= r + \gamma (1 - done)\, Q'(s', a; \theta') \\
\end{align*}
$$

In [113]:
@torch.no_grad()
def compute_targets(rewards, have_no_next_action, target_q_vals, gamma):
    return rewards + gamma * (1 - have_no_next_action) * target_q_vals

$$
\begin{align*}
\mathcal{L}(\theta) &= \frac{1}{2}\left( Q(s, a; \theta) - y \right)^{2}
\end{align*}
$$

In [114]:
def compute_loss(online_q_vals, targets):
    return (0.5 * (online_q_vals - targets).pow(2)).mean()

In [115]:
class DDQNUpdater:
    def __init__(self, online_q_network, target_q_network, optimizer):
        self.online_q_network = online_q_network
        self.target_q_network = target_q_network
        self.optimizer = optimizer

    def update_online_q_net(self, loss):
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

    def update_target_q_net(self):
        self.target_q_network.load_state_dict(
            self.online_q_network.state_dict()
        )

In [116]:
from collections import deque

In [117]:
class ReplayDeque:
    def __init__(self, maxlen):
        self.deque = deque(maxlen = maxlen)

    def append(self, state, action, next_state, reward, has_no_next_action):
        self.deque.append(
            (
                np.array(state, copy = False),
                action,
                np.array(next_state, copy = False),
                reward,
                has_no_next_action
            )
        )

    def sample(self, batch_size):
        states, actions, next_states, rewards, have_no_next_action = zip(*random.sample(self.deque, batch_size))

        return (
            np.stack(states),
            np.array(actions),
            np.stack(next_states),
            np.array(rewards, dtype = np.float32),
            np.array(have_no_next_action, dtype = np.float32)
        )

    def __len__(self):
        return len(self.deque)

In [118]:
class DDQNTrainer:
    def __init__(self, online_q_network, target_q_network, updater, replay_buffer, gamma, batch_size):
        self.online_q_network = online_q_network
        self.target_q_network = target_q_network
        self.updater = updater
        self.replay_buffer = replay_buffer
        self.gamma = gamma
        self.batch_size = batch_size

    def train_step(self):
        # 经验池不够，跳过（你之前说暂不判断，这里给出标准写法，你可删）
        if len(self.replay_buffer) < self.batch_size:
            return None

        # 采样 batch
        states, actions, next_states, rewards, dones = self.replay_buffer.sample(self.batch_size)

        # numpy → tensor
        states = torch.tensor(states, dtype=torch.float32)
        actions = torch.tensor(actions, dtype=torch.long).unsqueeze(1)
        next_states = torch.tensor(next_states, dtype=torch.float32)
        rewards = torch.tensor(rewards, dtype=torch.float32).unsqueeze(1)
        dones = torch.tensor(dones, dtype=torch.float32).unsqueeze(1)

        # Q(s, a; θ)
        q_vals = q_vals_for_states(self.online_q_network, states)
        q_vals_for_taken_actions = q_vals_for_actions(q_vals, actions)

        # Double-DQN target
        # Step 1: online network 做 argmax
        next_q_vals_online = q_vals_for_states(self.online_q_network, next_states)
        next_actions = actions_for_q_vals(next_q_vals_online)

        # Step 2: target network 取 Q'(s', argmax)
        next_q_vals_target = q_vals_for_states(self.target_q_network, next_states)
        target_q_vals = q_vals_for_actions(next_q_vals_target, next_actions)

        # 计算最终 target y
        targets = compute_targets(rewards, dones, target_q_vals, self.gamma)

        # 损失
        loss = compute_loss(q_vals_for_taken_actions, targets)

        # 更新 online
        self.updater.update_online_q_net(loss)

        return loss.item()


# 训练

# 可视化